In [40]:
import pyBrellaSampling.Code.Tools.io as io
import pyBrellaSampling.Code.Tools.classes as classes
import pyBrellaSampling.Code.Tools.QM.pyscf_tools as pyscf_tools

import sys
import os
from pprint import pprint

In [41]:
inputFilename = "qmmm_0.input"
inpFile = io.textRead(inputFilename)
nat = int(inpFile[0].split()[0])
ncharges = int(inpFile[0].split()[1])
atoms = [str]*nat
charge = int(0)
spin = int(0)
method = "PBE"
basis = "6-31G*"
for index, line in enumerate(inpFile[1:nat+1]):
    words = line.split()
    atoms[index] = f"{words[3]} {words[0]} {words[1]} {words[2]}"
charges = [float]*ncharges
charge_loc = [tuple]*ncharges
for i, line in enumerate(inpFile[nat+1:]):
    words = line.split()
    charges[i] = float(words[0]) #### WARNING, this should be 3. changed for testing!
    charge_loc[i] = (float(words[1]), float(words[2]), float(words[3]))
io.textDump(charges, "charges")
mol = pyscf_tools.genMol(atoms, charge, spin, basis,True)
if method.casefold() == "hf":
    mf, grad = pyscf_tools.UHF(mol, charges, charge_loc)
    # pprint(vars(mf.mm_mol))
else:
    mf, grad = pyscf_tools.DFT(mol, method, "None", True, 8, False, charges, charge_loc)
muliken, dipole = mf.analyze()
pprint(muliken[1])
# pprint(vars(mf))
result = [str]*(nat+ncharges+1)
result[0] = f"{mf.e_tot} {ncharges}"
for i in range(nat):
    result[i+1] = f"{grad[i][0]} {grad[i][1]} {grad[i][2]}"
for i in range(ncharges):
    result[i+nat] = ""

INFO Symetry is currently at C1
converged SCF energy = -355.371464081001  <S^2> = 1.5930013e-10  2S+1 = 1
--------------- QMMMUKS gradients ---------------
         x                y                z
0 F    -0.0047588831    -0.0084143597    -0.0113395922
1 H     0.0133529129    -0.0155422842     0.0020850178
2 C    -0.0267541707     0.0286477827    -0.0035031823
3 N     0.0168316422    -0.0175707400    -0.0002225446
4 Na     0.0065989572     0.0137633022     0.0114874997
----------------------------------------------
**** MO energy ****
                             alpha | beta                alpha | beta
MO #1   energy= -37.9285266265447  | -37.9285266265667  occ= 1 | 1
MO #2   energy= -24.0469614605131  | -24.046961445124   occ= 1 | 1
MO #3   energy= -14.0251502711956  | -14.0251510526006  occ= 1 | 1
MO #4   energy= -9.94337400222141  | -9.94337336073026  occ= 1 | 1
MO #5   energy= -2.04878495946896  | -2.04878495962747  occ= 1 | 1
MO #6   energy= -1.02583385511367  | -1.02583385542

In [44]:
import pyscf
from pyscf import qmmm, grad

In [53]:
dft_grad = grad.UKS(mf)
pprint((dft_grad))

pc = qmmm.QMMMGrad(dft_grad)
print(pc)
pprint(vars(pc))
# qmmm.mm_charge_grad(dft_grad, charge_loc, charges)

{'atmlst': None,
 'base': <pyscf.qmmm.itrf.QMMMUKS object at 0x764fa916bb10>,
 'de': None,
 'grids': None,
 'max_memory': 4000,
 'mol': <pyscf.gto.mole.Mole object at 0x76502432b390>,
 'nlcgrids': None,
 'stdout': <ipykernel.iostream.OutStream object at 0x7650286fc730>,
 'unit': 'au',
 'verbose': 3}


In [54]:
pc_grad = pc.grad_nuc_mm()

In [57]:
print(pc_grad[0])

[ 9.86770287e-06  4.91219310e-06 -4.03831995e-05]
